# 01a keithley acq waveform


# Preamble

## Preamble for Jupyter

In [ ]:
%load_ext autoreload
%autoreload 2

## Preamble Main Libraries and a few usefull constants

In [ ]:
import sys  # noqa: F401
import os  # noqa: F401

import numpy as np
import time  # noqa: F401
from datetime import datetime

import plotly.graph_objects as go


from scipy import constants

deg2rad = np.deg2rad(1)
rad2deg = np.rad2deg(1)
eps = np.finfo(np.float64).eps
hc = constants.value("inverse meter-electron volt relationship")

In [ ]:
import keithley_utils as kthu  # your serial helper module;
from keithley_utils import KeithleyPicoammeter  # your serial helper module;

from wg_toolkit.misc import datenow_str  # type: ignore
from wg_toolkit.logprint import printc, print_info, print_warning, print_error, print_log, print_done, print_success  # type: ignore # noqa: F401

print_done(f"Notebook started at {datenow_str()}")

print_success("#" * 50)

## Preamble Plotly

In [ ]:
import plotly.colors

colors_plotly = plotly.colors.DEFAULT_PLOTLY_COLORS


def custom_layout(xlabel=None, ylabel=None, title=None, template="seaborn"):
    return go.Layout(
        template=template,  # Set the template
        width=800,  # Set the figure width
        height=600,  # Set the figure height
        font=dict(family="Georgia", size=14, color="#3B3B3B"),  # General font settings
        title=dict(
            text=title,
            font=dict(size=24),
            x=0.5,
            xanchor="center",
        ),
        xaxis_title=dict(text=xlabel, font=dict(size=18)),
        yaxis_title=dict(text=ylabel, font=dict(size=18)),
        xaxis=dict(tickfont=dict(size=16)),  # Set X-axis tick font size
        yaxis=dict(tickfont=dict(size=16)),  # Set Y-axis tick font size
    )


# initialize figures with something like
# plotly_fig = go.Figure(layout=custom_layout("Frequency [Hz]", "Amplitude [Volts]", fname))


## Local Constants, variables and functions

In [ ]:
POWER_LINE_FREQ = 60.0  # set to 50.0 if on 50 Hz mains
POWER_LINE_PERIOD = 1 / POWER_LINE_FREQ

In [ ]:
save_plot = False
close_connections_at_end = False
DEBUG = False

# Main

## Detect Keithley Devices

In [ ]:
devs = kthu.detect_keithley_devices(baudrate=None, verbose=True)

In [ ]:
devs

In [ ]:
# Goal is to stop notebook execution if no Keithley devices are detected, or if the detected device is not a Keithley instrument. This prevents further errors down the line when trying to communicate with the device.
if not devs:
    print_error("No Keithley devices detected. Please check connections and try again.")
    sys.exit(1)

dev = devs[0]  # Assuming we want to use the first detected device
if not dev["is_keithley"]:
    print_error("The detected device is not a Keithley instrument. MOVING ON")
    sys.exit(1)


### Select serial port

In [ ]:
SERIALPORT = dev["port"]  # type: ignore

if SERIALPORT is None:
    print_error("No valid serial port found for device.")
else:
    print_info(f"Using serial port: {SERIALPORT}")

In [ ]:
kth_dev = KeithleyPicoammeter(SERIALPORT, verbose=True, debug=False)
vars(kth_dev)

## Clear Buffer

In [ ]:
buffer_df = {}  # to store results from multiple acquisitions if desired. Created earlier on to avoid reinitialization in a loop

## Reset Instrument and Check for Errors

In [ ]:
_ = kth_dev.reset_instrument()

## Setup Acquisition Parameters

In [ ]:
NPLC_SP = 0.01000
NUM_POINTS = 100

if NUM_POINTS * NPLC_SP * POWER_LINE_PERIOD > 2.01:
    NUM_POINTS = int(1.0 / (NPLC_SP * POWER_LINE_PERIOD))
    print(
        f"[WARNING] Adjusted NUM_POINTS to {NUM_POINTS} to ensure acquisition time is under 2 seconds for NPLC={NPLC_SP} and power line frequency={POWER_LINE_FREQ} Hz"
    )
# act_time = 0.500
# NUM_POINTS = int(act_time / (NPLC_SP * POWER_LINE_PERIOD))
# NOTE: Buffer stores up to 3000 readings
# CURR_RANGE = None  # None for autorange, or set to a specific range in amps (e.g. 1e-6 for 1 uA range)
CURR_RANGE = 1e-9
# smallest range is 2nA, and highest range is 20mA for the Keithley 6487.
# if you try a  arbitrary range, they instrument will set the closest available range.

print(f"[INFO] Set acquisition parameters: NPLC = {NPLC_SP}, NUM_POINTS = {NUM_POINTS}, CURR_RANGE = {CURR_RANGE} A")

cal_acq_time = NUM_POINTS * NPLC_SP * POWER_LINE_PERIOD
print(f"[INFO] Calculated acquisition time: {cal_acq_time:.3f} seconds")

calc_integration_time = NPLC_SP * POWER_LINE_PERIOD
print(f"[INFO] Calculated integration time: {calc_integration_time * 1e3:.3f} ms\n\n\n")


actual_range, actual_nplc = kth_dev.set_range(set_curr_range=CURR_RANGE, nplc=NPLC_SP)

print_info(f"Actual settings after configuration: NPLC = {actual_nplc}, Current Range = {actual_range} A")


In [ ]:
actual_acq_time = NUM_POINTS * actual_nplc * POWER_LINE_PERIOD
actual_integration_time = actual_nplc * POWER_LINE_PERIOD
acq_rate_SP = 1 / NPLC_SP / POWER_LINE_PERIOD

print("\n[INFO] Set Points and actual values for acquisition:")

print(f"\tRange SP:\t\t\t{CURR_RANGE:.6g} A") if CURR_RANGE is not None else print("\tRange SP:\t\t\tAUTO RANGE")
print(f"\tRange Actual:\t\t\t{actual_range:.6g} A")
print(f"\tNUM_POINTS:\t\t\t{NUM_POINTS} points")
print(f"\tNPLC SP:\t\t\t{NPLC_SP:.3f}")
print(f"\tNPLC Actual:\t\t\t{actual_nplc:.3f}")
print(f"\tIntegration Time SP:\t\t{calc_integration_time * 1e3:.3f} ms")
print(f"\tIntegration Time Actual:\t{actual_integration_time * 1e3:.3f} ms")
print(f"\tCalc Acq Time SP:\t\t{cal_acq_time * 1e3:.3f} ms")
print(f"\tActual Acq Time:\t\t{actual_acq_time * 1e3:.3f} ms")

## Zero Instrument and Check for Errors

In [ ]:
kth_dev.check_inst_errors()

In [ ]:
# %% Zero Instrument and Check for Errors
zero_val = kth_dev.zero_instrument()

## Setup Waveform Acquisition

Mostly configure the buffer and the trigger settings. The actual acquisition is done in the next cell to allow for multiple acquisitions without reconfiguring the instrument each time.

In [ ]:
# Setup Acquisition.
kth_dev.setup_waveform_acquisition(
    num_points=NUM_POINTS,
)


## Acquire Waveform, parse data, and load to buffer variable

In [ ]:
# Low level commands for troubleshooting acquisition setup and buffer state.

# kth_dev.query_and_check("*OPC?", )
# # kth_dev.query_and_check(":TRAC:CLE", )
# kth_dev.query_and_check("TRAC:FEED?", )
# kth_dev.query_and_check("TRAC:FEED:CONT?", )
# # kth_dev.query_and_check(":TRAC:FEED:CONT NEXT", )
# resp = kth_dev.query_and_check(":TRAC:POIN:ACT?", )
# print(f"[DEBUG] : Current points in buffer: {resp}, {type(resp) = }")


In [ ]:
# buffer_df = {}  # to store results from multiple acquisitions if desired. Created earlier on to avoid reinitialization in a loop

In [ ]:
kth_dev.debug = False  # enable debug mode to print instrument responses for troubleshooting

In [ ]:
for _ in range(1):
    df_acq = kth_dev.acq_waveform(poll_interval=0.1, timeout=10.0)

    # Data parsing and loading to buffer
    df_acq["Time_msecs"] = df_acq["Time_Secs"] * 1000
    df_acq.attrs["NPLC"] = actual_nplc
    df_acq.attrs["Range"] = actual_range

    df_acq.attrs["acq_rate_SP"] = 1 / df_acq.attrs["NPLC"] / POWER_LINE_PERIOD
    df_acq.attrs["actual_acq_rate"] = 1 / df_acq["Time_Secs"].diff().dropna().mean()

    print_info(f"Acquired samples: {len(df_acq)}")
    print_done("Parsing Data DONE...")

    print_info("Loading DataFrame to a Buffer dictionary...")
    buffer_df[datenow_str()] = df_acq.copy()  # store in buffer if doing multiple acquisitions in a loop
    print_done(f"Buffer contain {len(buffer_df)} acquisition(s).")
    printc(f"\nLast run ID: {sorted(buffer_df.keys())[-1]}", color="gray", bold=True)

    # kth_dev.query_and_check(
    #     ":TRAC:CLE",
    # )

print("\n")
print_success("#" * 50)
print_success("All acquisitions done.")
print_success("#" * 50)

In [ ]:
ids = sorted(buffer_df.keys())  # run ids

print("\n[SUMMARY] Info for all acquisitions:")
for _id in ids:
    _acq_rate_SP = 1 / buffer_df[_id].attrs["NPLC"] / POWER_LINE_PERIOD
    _actual_acq_rate = buffer_df[_id].attrs["actual_acq_rate"]
    print(
        f"id = {_id}, Npoints = {len(buffer_df[_id])}, NPLC = {buffer_df[_id].attrs['NPLC']}, Range = {buffer_df[_id].attrs['Range']}, Acq Rate SP = {_acq_rate_SP:.2f}, Mean Actual Acq Rate = {_actual_acq_rate:.2f} samples/sec"
    )

### Check for repeated waveforms
Make sure buffer was cleared before each acquisition. If not, the new data will be appended to the old data in the buffer and cause confusion when parsing and loading to dataframe.

In [ ]:
# Compare all Current_Amps data between all acquisitions
print("\n[VALIDATION] Comparing Current_Amps between all acquisitions:")

duplicates_found = False

# Compare all pairs
for i in range(len(ids)):
    for j in range(i + 1, len(ids)):
        run_id_1 = ids[i]
        run_id_2 = ids[j]

        if buffer_df[run_id_1].shape[0] == buffer_df[run_id_2].shape[0]:
            try:
                # Boolean comparison
                are_equal = (buffer_df[run_id_1]["Current_Amps"] == buffer_df[run_id_2]["Current_Amps"]).all()
            except Exception as e:
                print(f"[ERROR] Comparison failed between {run_id_1} and {run_id_2}: {e}")
                continue
        else:
            are_equal = False

        if are_equal:
            print(f"[WARNING] DUPLICATE: {run_id_1} == {run_id_2} -> {are_equal}")
            duplicates_found = True
        else:
            print(f"[INFO] {run_id_1} == {run_id_2} -> {are_equal}")

if duplicates_found:
    print_warning("Some acquisitions contain IDENTICAL data!")
else:
    print_success("All acquisitions contain DIFFERENT data!")

### Review Results

In [ ]:
ids

In [ ]:
print_success("[RESULTS] LAST ACQUISITION DataFrame info:")
print(buffer_df[ids[-1]].info())
print_success("[RESULTS] LAST ACQUISITION DataFrame describe:")
print(buffer_df[ids[-1]].describe())


# Post Processing

## Load last Acquisition for Post Processing

In [ ]:
print("\n[POSTPROCESSING] Post Processing STARTED")

print("\n[POSTPROCESSING] Plotting data with Plotly...")


plotly_fig = go.Figure(layout=custom_layout("Time (s)", "Current (A)", "Keithley Data Acquisition"))
# fig = go.Figure()

_col_x = "Time_Secs"
_col_y = "Current_Amps"

_id2plot = ids[-1]
_id2plot = ids[-1:-105:-1]  # show the most recent runs

for _id in _id2plot:
    df = buffer_df[_id]
    print(f"[POSTPROCESSING] Plotting Run ID: {_id} with {len(df)} samples...")

    _label = f"{_id}, N={len(df)}, NPLC={df.attrs['NPLC']:.2f}"

    plotly_fig.add_trace(go.Scatter(x=df[_col_x], y=df[_col_y], mode="lines+markers", name=_label))

    plotly_fig.update_layout(
        hovermode="x unified",
        width=1200,
        height=600,
    )
    # Save HTML and open in VS Code webview
    html_file = f"Results\\{datenow_str()}_keithley_acquisition_plot.html"

    if save_plot:
        plotly_fig.write_html(html_file)
        print(f"[POSTPROCESSING] Plot saved to .\\{html_file}\n")

plotly_fig.show()